In [13]:
import time

notebook_start_time = time.time()
print("[Notebook timer started]")

[Notebook timer started]
[Cell executed in 0.002 seconds]


# Redoing some analysis to align with SQL analysis
## Time import 
Now we can see how long each cell takes to run

In [14]:

from IPython import get_ipython

_cell_start_time = None


def _pre_run_cell(info):
    global _cell_start_time
    _cell_start_time = time.time()


def _post_run_cell(result):
    global _cell_start_time
    if _cell_start_time is None:
        return
    elapsed = time.time() - _cell_start_time
    print(f"[Cell executed in {elapsed:.3f} seconds]")


_ip = get_ipython()
if _ip is not None:
    _ip.events.register("pre_run_cell", _pre_run_cell)
    _ip.events.register("post_run_cell", _post_run_cell)
else:
    print("[Timing hooks not installed: no active IPython shell detected]")


## Data Import 
Importing small sample of data to use as a prototype. Here, we will be using python with the pandas library to attain a performance baseline.

In [26]:
import pandas as pd

df = pd.read_csv("Prototype/trips_prototype.csv")
#df = pd.read_csv("C:\year3-bigData-proj\project\Cyclistic_Data\datasets\cyclistic_tripdata_2021.csv")
df.tail()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
148040,1CA9149F54BE9D87,classic_bike,2021-05-08 12:38:24,2021-05-08 12:47:37,Halsted St & Wrightwood Ave,TA1309000061,Halsted St & Clybourn Ave,331,41.929143,-87.649077,41.909668,-87.648128,member
148041,5D7E4486D664ED3B,electric_bike,2021-05-28 10:33:25,2021-05-28 10:42:28,NaN,NaN,Ashland Ave & Lake St,13073,41.890000,-87.700000,41.886683,-87.667239,member
148042,44832A3A574FF1EF,docked_bike,2020-11-07 17:55:00,2020-11-07 17:59:53,Larrabee St & Webster Ave,144.0,Bissell St & Armitage Ave,113.0,41.921822,-87.644140,41.918440,-87.652220,casual
148043,034DBA441C0D1A44,classic_bike,2022-10-03 17:10:15,2022-10-03 17:19:04,Lincoln Ave & Winona St,KA1504000078,Ravenswood Ave & Lawrence Ave,TA1309000066,41.974911,-87.692503,41.969090,-87.674237,member
148044,174BB0AF9E73B087,classic_bike,2021-07-24 15:28:07,2021-07-24 15:42:43,Sangamon St & Washington Blvd,13409,Michigan Ave & Madison St,13036,41.883165,-87.651100,41.882134,-87.625125,casual


[Cell executed in 1.036 seconds]
[Cell executed in 1.036 seconds]


In [16]:
df["started_at"] = pd.to_datetime(df["started_at"], utc=True)
df["ended_at"] = pd.to_datetime(df["ended_at"], utc=True)
df["ride_length"] = df["ended_at"] - df["started_at"]
df["ride_length_minutes"] = df["ride_length"].dt.total_seconds() / 60
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_length,ride_length_minutes
0,53360EC85A03516C,electric_bike,2020-11-05 15:50:30+00:00,2020-11-05 16:08:25+00:00,NaN,NaN,Western Ave & Congress Pkwy,382.0,41.930000,-87.710000,41.874704,-87.686426,casual,0 days 00:17:55,17.916667
1,05E0A2BD796D7CEE,classic_bike,2022-06-26 12:11:44+00:00,2022-06-26 13:19:51+00:00,Broadway & Argyle St,13108,Sheridan Rd & Montrose Ave,TA1307000107,41.973815,-87.659660,41.961670,-87.654640,casual,0 days 01:08:07,68.116667
2,3B995854F4E5F068,classic_bike,2022-09-17 23:50:22+00:00,2022-09-18 00:01:37+00:00,Wells St & Concord Ln,TA1308000050,Clark St & Elm St,TA1307000039,41.912133,-87.634656,41.902973,-87.631280,member,0 days 00:11:15,11.250000
3,03A33065C5122AC4,electric_bike,2021-10-29 19:50:05+00:00,2021-10-29 19:56:11+00:00,Clark St & Chicago Ave,13303,Michigan Ave & Lake St,TA1305000011,41.896686,-87.630788,41.886395,-87.624700,member,0 days 00:06:06,6.100000
4,9B8164B9E681B53F,classic_bike,2022-05-03 22:46:04+00:00,2022-05-03 22:54:49+00:00,Milwaukee Ave & Grand Ave,13033,Fairbanks Ct & Grand Ave,TA1305000003,41.891578,-87.648384,41.891847,-87.620580,casual,0 days 00:08:45,8.750000


[Cell executed in 0.122 seconds]
[Cell executed in 0.122 seconds]


## Making Time and Day cols
Allows for us to perform day and time analysis

In [17]:
df["hour"] = df["started_at"].dt.hour
df["day_of_week"] = df["started_at"].dt.day_name()

df[["started_at", "hour", "day_of_week"]].head()

,started_at,hour,day_of_week
0,2020-11-05 15:50:30+00:00,15,Thursday
1,2022-06-26 12:11:44+00:00,12,Sunday
2,2022-09-17 23:50:22+00:00,23,Saturday
3,2021-10-29 19:50:05+00:00,19,Friday
4,2022-05-03 22:46:04+00:00,22,Tuesday


[Cell executed in 0.039 seconds]
[Cell executed in 0.039 seconds]


## Bike type usage
Here, we wanted to find the relationship between bike type and user type. We found the average ride duration for each bike and user combination. This analysis helps identify user preferences for specific bike types and highlights the differences between members and casual riders that we have previously theorised about. 
The shear number of member usage is higher, but when we look at the percentage usage, the figures start to make sense. Members seem to prefer the "classic bike" over the electric more so than the casual users do, this could be down to members being into fitness or simply the bike availability in their residential areas.
The only question we are left with right now is why do only casual users use "docked bikes"?

In [18]:
bike_user_ct = pd.crosstab(df["rideable_type"], df["member_casual"])
print("Bike type x user type (counts):")
print(bike_user_ct)


print("\nBike type x user type (row %):")
bike_pct = bike_user_ct.div(bike_user_ct.sum(axis=1), axis=0).round(3)
print(bike_pct)

print("\nAverage ride length (minutes) by bike type and user type:")

avg_duration_bike_user = (
    df.groupby(["rideable_type", "member_casual"])["ride_length_minutes"]
    .mean()
    .round(1)
    .unstack()
)
print(avg_duration_bike_user)

print("\nInterpretation by bike type:")
for bike in bike_user_ct.index:
    casual_share = bike_pct.loc[bike, "casual"] * 100
    member_share = bike_pct.loc[bike, "member"] * 100
    print(f"- {bike}: {casual_share:.1f}% casual, {member_share:.1f}% member")

Bike type x user type (counts):
member_casual  casual  member
rideable_type                
classic_bike    21765   37626
docked_bike     16284   18194
electric_bike   23996   30180

Bike type x user type (row %):
member_casual  casual  member
rideable_type                
classic_bike    0.366   0.634
docked_bike     0.472   0.528
electric_bike   0.443   0.557

Average ride length (minutes) by bike type and user type:
member_casual  casual  member
rideable_type                
classic_bike     29.0    14.0
docked_bike      57.7    12.5
electric_bike    18.0    12.1

Interpretation by bike type:
- classic_bike: 36.6% casual, 63.4% member
- docked_bike: 47.2% casual, 52.8% member
- electric_bike: 44.3% casual, 55.7% member
[Cell executed in 0.143 seconds]
[Cell executed in 0.143 seconds]


## Station-based insights
We wanted to find out which stations had the highest numbers of users arriving and departing from them, we can see that "Dearborn St and Erie St" appears one of the focal points, seeing as it is the most popular destination and second most popular starting point. Similar observations can be made for each other station.
We also computed the average ride duration by start station an user type. We filtered this to include stations with 30+ trips to ensure reliability.
Using this information, we can detarmine what areas are residential and which are tourist hotspots among many other things.

In [19]:
for user_type in ["member", "casual"]:
    print(f"\nTop 10 start stations for {user_type}s:")
    top_start = (
        df[df["member_casual"] == user_type]["start_station_name"]
        .value_counts()
        .head(10)
    )
    print(top_start)

    print(f"\nTop 10 end stations for {user_type}s:")
    top_end = (
        df[df["member_casual"] == user_type]["end_station_name"]
        .value_counts()
        .head(10)
    )
    print(top_end)

min_trips_per_station = 30

avg_duration_station = (
    df.groupby(["start_station_name", "member_casual"])["ride_length_minutes"]
    .agg(["count", "mean"])
    .reset_index()
)



avg_duration_station = avg_duration_station[avg_duration_station["count"] >= min_trips_per_station]

print("\nAverage ride length (minutes) by start station and user type (>= 30 trips):")
print(avg_duration_station.sort_values("mean", ascending=False).head(20))


Top 10 start stations for members:
start_station_name
Clark St & Elm St               693
Kingsbury St & Kinzie St        630
Wells St & Concord Ln           594
Wells St & Elm St               562
Dearborn St & Erie St           526
St. Clair St & Erie St          519
Broadway & Barry Ave            492
Clinton St & Madison St         485
Wells St & Huron St             483
Clinton St & Washington Blvd    467
Name: count, dtype: int64

Top 10 end stations for members:
end_station_name
Clark St & Elm St               677
Wells St & Concord Ln           637
Kingsbury St & Kinzie St        603
Broadway & Barry Ave            558
Wells St & Elm St               543
St. Clair St & Erie St          523
Clinton St & Madison St         523
Dearborn St & Erie St           521
Clinton St & Washington Blvd    489
Wells St & Huron St             468
Name: count, dtype: int64

Top 10 start stations for casuals:
start_station_name
Streeter Dr & Grand Ave              1503
Millennium Park          

## Time of day x day of week bike usage
We could use seaborn to display this information on a heatmap as it is very suited for that, but seeing as we are attempting to set a base line for time, we thought it best to stick with text outputs in case graphics take a significantly long amount of time to display.
Now bike usage by day of week and time is legible. The huge numbers of users visible during the work rush hours on weekdays confirms our earlier hypotheses. The low numbers of rides over the weekend do so as well, likely being mostly tourist use.

In [20]:
#this is basically a text heat map, I wanted to make something easily duplicated in other languages without importing libraries
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
cat_type = pd.CategoricalDtype(categories=day_order, ordered=True)
df["day_of_week"] = df["day_of_week"].astype(cat_type)

for user_type in ["member", "casual"]:
    subset = df[df["member_casual"] == user_type]
    pivot = subset.pivot_table(
        index="day_of_week",
        columns="hour",
        values="ride_id",
        aggfunc="count",
        fill_value=0,
    )

    print(f"\nTrips table (day x hour) for {user_type}:")
    print(pivot)

   


Trips table (day x hour) for member:
hour          0    1   2   3   4    5    6    7    8    9   ...   14   15  \
day_of_week                                                 ...             
Monday        66   36  33  23  26  129  363  729  827  530  ...  601  734   
Tuesday       51   27  15  15  31  165  475  865  950  517  ...  619  780   
Wednesday     58   30  17   8  28  155  452  907  992  571  ...  623  747   
Thursday      99   44  20  17  27  148  432  804  911  581  ...  648  758   
Friday       104   57  28  24  37  145  365  641  690  497  ...  743  863   
Saturday     202  137  94  44  26   45  139  243  410  626  ...  949  945   
Sunday       203  173  88  50  36   43  113  179  259  434  ...  867  864   

hour           16    17    18   19   20   21   22   23  
day_of_week                                             
Monday       1103  1395  1176  795  527  364  206  115  
Tuesday      1214  1519  1278  869  551  388  244  139  
Wednesday    1190  1578  1311  887  605 

## Analysing Daily Trends
we can see saturday is extra busy for casual users which supports the theory they are tourists trying to view the city

In [21]:
weekday_counts = (
    df.groupby(["day_of_week", "member_casual"])
    .size()
    .unstack(fill_value=0)
    .reindex(day_order)
)

member_counts = weekday_counts["member"]
casual_counts = weekday_counts["casual"]


print("Trips by day of week and user type (counts):")
print(weekday_counts)


Trips by day of week and user type (counts):
member_casual  casual  member
day_of_week                  
Monday           7018   11923
Tuesday          6701   12967
Wednesday        6960   13276
Thursday         7495   13149
Friday           9045   12352
Saturday        13790   11982
Sunday          11036   10351
[Cell executed in 0.019 seconds]
[Cell executed in 0.019 seconds]


## Busy Stations
We can see the busiest stations, ranked on the number of arrivals + departures

In [22]:
departures = df.groupby("start_station_name").size().rename("departures")
arrivals = df.groupby("end_station_name").size().rename("arrivals")


station_activity = (
    departures.to_frame()
    .join(arrivals.to_frame(), how="outer")
    .fillna(0)
)

station_activity["total_activity"] = (
    station_activity["departures"] + station_activity["arrivals"]
)



top_stations = station_activity.sort_values("total_activity", ascending=False).head(10)

print("Top stations by total activity (departures + arrivals):")
print(top_stations)

Top stations by total activity (departures + arrivals):
                            departures  arrivals  total_activity
start_station_name                                              
Streeter Dr & Grand Ave         1916.0    1952.0          3868.0
Michigan Ave & Oak St           1151.0    1114.0          2265.0
Clark St & Elm St               1107.0    1083.0          2190.0
Wells St & Concord Ln           1063.0    1097.0          2160.0
Millennium Park                 1047.0    1062.0          2109.0
Theater on the Lake              974.0    1007.0          1981.0
Wells St & Elm St                924.0     907.0          1831.0
Clark St & Lincoln Ave           894.0     847.0          1741.0
Indiana Ave & Roosevelt Rd       905.0     811.0          1716.0
Broadway & Barry Ave             805.0     901.0          1706.0
[Cell executed in 0.038 seconds]
[Cell executed in 0.038 seconds]


## Ride lengths by bike type
We can see what the average ride length is for both casuals and members for each type of bike.

In [23]:

valid = df[df["ride_length_minutes"].notna()].copy()

avg_duration_bike_user = (valid.groupby(["rideable_type", "member_casual"])["ride_length_minutes"].mean().round(1).unstack())

print("Average ride length (minutes) by bike type and user type:")
print(avg_duration_bike_user)


Average ride length (minutes) by bike type and user type:
member_casual  casual  member
rideable_type                
classic_bike     29.0    14.0
docked_bike      57.7    12.5
electric_bike    18.0    12.1
[Cell executed in 0.047 seconds]
[Cell executed in 0.048 seconds]


In [24]:


elapsed = time.time() - notebook_start_time
print(f"[Whole notebook executed in {elapsed:.3f} seconds]")

[Whole notebook executed in 1.252 seconds]
[Cell executed in 0.003 seconds]
[Cell executed in 0.003 seconds]
